# Setup and Settings

In [ ]:
from pathlib import Path

import pandas as pd

project_root = next(
    (
        path
        for path in [Path.cwd(), *Path.cwd().parents]
        if (path / "pyproject.toml").is_file()
    ),
    Path.cwd(),
)

source_path = project_root / r"path/to/file.xlsb"
sheet = 0
csv_separator = ","
csv_encoding = "utf-8-sig"

# Load and Preview

In [ ]:
suffix = source_path.suffix.lower()
if suffix == ".csv":
    df = pd.read_csv(
        source_path,
        dtype=str,
        keep_default_na=False,
        sep=csv_separator,
        encoding=csv_encoding,
    )
elif suffix in {".xls", ".xlsx", ".xlsm", ".xlsb", ".ods"}:
    # pandas-stubs references the unavailable optional pyxlsb workbook type.
    df = pd.read_excel(  # pyright: ignore[reportUnknownMemberType]
        source_path,
        sheet_name=sheet,
        engine="calamine",
        dtype=str,
        keep_default_na=False,
    )
else:
    raise ValueError(f"Unsupported input format: {suffix}")
df

# Save Raw Parquet

In [ ]:
output_dir = project_root / "data/raw/parquet"
output_dir.mkdir(parents=True, exist_ok=True)
output_path = output_dir / f"{source_path.stem}.parquet"
with output_path.open("xb") as output_file:
    df.to_parquet(output_file, engine="pyarrow", index=False)